# **Langkah 2: Melatih Model Baseline (8 Kategori) Multiclass**

###**2.1 Definisi Fungsi Pembantu - Memory Management (Helper Functions)**

In [ ]:
def get_memory_usage():
    """Mendapatkan penggunaan RAM sistem"""
    memory = psutil.virtual_memory()
    used_gb = (memory.total - memory.available) / (1024**3)
    total_gb = memory.total / (1024**3)
    available_gb = memory.available / (1024**3)
    percent = memory.percent
    return used_gb, total_gb, available_gb, percent

def print_memory_status(stage=""):
    """Print status memori"""
    used, total, available, percent = get_memory_usage()
    print(f"\n[Memory Status - {stage}]")
    print(f"  Used: {used:.2f} GB / {total:.2f} GB ({percent:.1f}%)")
    print(f"  Available: {available:.2f} GB")

def clear_memory():
    """Bersihkan memory"""
    gc.collect()
    print("  Memory cleared")

print(" Memory management functions defined!")

 Memory management functions defined!


###**2.2 Definisi Fungsi Evaluasi Model**

In [ ]:
def evaluate_model(y_true, y_pred, model_name, training_time):
    """
    Fungsi untuk evaluasi model dengan berbagai metrik

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    model_name : str
        Nama model
    training_time : float
        Waktu training dalam detik

    Returns:
    --------
    results : dict
        Dictionary berisi semua metrik evaluasi
    cm : array
        Confusion matrix
    report : dict
        Classification report detail per class
    """

    # Calculate all metrics
    results = {
        'model_name': model_name,
        'dataset_type': 'Full',
        'training_time_seconds': training_time,

        # Accuracy
        'accuracy': accuracy_score(y_true, y_pred),

        # Precision (3 averaging methods)
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),

        # Recall (3 averaging methods)
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),

        # F1-Score (3 averaging methods)
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),

        # Matthews Correlation Coefficient
        'mcc': matthews_corrcoef(y_true, y_pred)
    }

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    # Classification Report (detail per class)
    class_names = le.classes_
    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    return results, cm, report

print("evaluate_model() function defined!")

evaluate_model() function defined!


###**2.3 Definisi Fungsi Print Results**

In [ ]:
def print_results(results):
    """
    Print hasil evaluasi dengan format yang rapi

    Parameters:
    -----------
    results : dict
        Dictionary hasil evaluasi dari evaluate_model()
    """
    print(f"\n{'='*70}")
    print(f"HASIL EVALUASI: {results['model_name']} ({results['dataset_type']})")
    print(f"{'='*70}")
    print(f"Waktu Training: {results['training_time_seconds']:.2f} detik ({results['training_time_seconds']/60:.2f} menit)")

    print(f"\nMetrik Utama:")
    print(f"  Accuracy          : {results['accuracy']:.4f}")

    print(f"\nPrecision:")
    print(f"  Macro             : {results['precision_macro']:.4f}")
    print(f"  Weighted          : {results['precision_weighted']:.4f}")
    print(f"  Micro             : {results['precision_micro']:.4f}")

    print(f"\nRecall:")
    print(f"  Macro             : {results['recall_macro']:.4f}")
    print(f"  Weighted          : {results['recall_weighted']:.4f}")
    print(f"  Micro             : {results['recall_micro']:.4f}")

    print(f"\nF1-Score:")
    print(f"  Macro             : {results['f1_macro']:.4f}")
    print(f"  Weighted          : {results['f1_weighted']:.4f}")
    print(f"  Micro             : {results['f1_micro']:.4f}")

    print(f"\nMCC                 : {results['mcc']:.4f}")
    print(f"{'='*70}")

print("print_results() function defined!")

print_results() function defined!


###**2.4 Definisi Fungsi Print Detailed Report**

In [ ]:
def print_detailed_report(y_true, y_pred, model_name, label_encoder):
    """
    Print detailed classification report per category dengan format tabel

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    model_name : str
        Nama model
    label_encoder : LabelEncoder
        Label encoder yang digunakan

    Returns:
    --------
    report_dict : dict
        Dictionary classification report
    """
    print("\n" + "="*80)
    print(f"DETAILED CLASSIFICATION REPORT: {model_name}")
    print("="*80)

    # Get category names
    target_names = label_encoder.classes_

    # Generate classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        digits=4,
        zero_division=0
    )

    print("\n" + report)

    # Calculate overall accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\nConfirmed Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

    # Parse report to dictionary
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    # Create detailed table view
    print("\n" + "="*80)
    print(f"PER-CATEGORY METRICS (Table Format)")
    print("="*80)
    print(f"{'Category':<15} {'Precision':>12} {'Recall':>12} {'F1-Score':>12} {'Support':>12}")
    print("-"*80)

    # Print each category
    for category in target_names:
        if category in report_dict:
            p = report_dict[category]['precision']
            r = report_dict[category]['recall']
            f1 = report_dict[category]['f1-score']
            support = int(report_dict[category]['support'])
            print(f"{category:<15} {p:>12.4f} {r:>12.4f} {f1:>12.4f} {support:>12,}")

    print("-"*80)

    # Print averages
    print(f"{'Macro Avg':<15} {report_dict['macro avg']['precision']:>12.4f} "
          f"{report_dict['macro avg']['recall']:>12.4f} "
          f"{report_dict['macro avg']['f1-score']:>12.4f} "
          f"{int(report_dict['macro avg']['support']):>12,}")

    print(f"{'Weighted Avg':<15} {report_dict['weighted avg']['precision']:>12.4f} "
          f"{report_dict['weighted avg']['recall']:>12.4f} "
          f"{report_dict['weighted avg']['f1-score']:>12.4f} "
          f"{int(report_dict['weighted avg']['support']):>12,}")

    print("="*80)

    return report_dict

print("print_detailed_report() function defined!")

print_detailed_report() function defined!


In [ ]:
# Inisialisasi List untuk Menyimpan Hasil
all_results = []
print("\n All helper functions ready!")
print("Results storage initialized!")


 All helper functions ready!
Results storage initialized!


###**2.5 Load Kembali Hasil Pre-Processing Data**

In [ ]:
print_memory_status("Sebelum Load Data")

print("\nMemuat data terproses dari Google Drive...")

# Load data
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

# Load label encoder
with open(os.path.join(PROCESSED_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

# Load feature names
with open(os.path.join(PROCESSED_DIR, 'feature_names.json'), 'r') as f:
    feature_names = json.load(f)

print(f"\nData dimuat:")
print(f"  - X_train shape: {X_train.shape}")
print(f"  - X_test shape: {X_test.shape}")
print(f"  - y_train shape: {y_train.shape}")
print(f"  - y_test shape: {y_test.shape}")
print(f"  - Jumlah kategori: {len(np.unique(y_train))}")
print(f"  - Jumlah fitur: {len(feature_names)}")

print(f"\n  Kategori: {le.classes_}")

print_memory_status("Setelah Load Data")


[Memory Status - Sebelum Load Data]
  Used: 2.07 GB / 12.67 GB (16.3%)
  Available: 10.60 GB

Memuat data terproses dari Google Drive...

Data dimuat:
  - X_train shape: (386274, 71)
  - X_test shape: (45319, 71)
  - y_train shape: (386274,)
  - y_test shape: (45319,)
  - Jumlah kategori: 8
  - Jumlah fitur: 71

  Kategori: ['Benign' 'Brute_Force' 'DDoS' 'DoS' 'MITM/Spoofing' 'Malware/Mirai'
 'Recon' 'Web_Based']

[Memory Status - Setelah Load Data]
  Used: 2.30 GB / 12.67 GB (18.1%)
  Available: 10.37 GB


###**2.6 Training Random Forest**

In [ ]:
print_memory_status("Sebelum Training RF")

start_time = time.time()

# Initialize Random Forest dengan parameter yang disesuaikan untuk dataset CICIIoT2025
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,           # Lebih shallow dari CICIoT2023
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Mulai training Random Forest...")
rf_model.fit(X_train, y_train)

rf_time = time.time() - start_time

print(f"\nTraining selesai dalam {rf_time:.2f} detik ({rf_time/60:.2f} menit)")

print("\nMelakukan prediksi pada test set...")
y_pred_rf = rf_model.predict(X_test)

print(" Prediksi selesai!")

print_memory_status("Setelah Training RF")


[Memory Status - Sebelum Training RF]
  Used: 2.55 GB / 12.67 GB (20.1%)
  Available: 10.12 GB
Mulai training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   36.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.6min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.1s



Training selesai dalam 94.82 detik (1.58 menit)

Melakukan prediksi pada test set...
 Prediksi selesai!

[Memory Status - Setelah Training RF]
  Used: 2.55 GB / 12.67 GB (20.1%)
  Available: 10.12 GB


[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.2s finished


In [ ]:
print("\nEvaluasi Random Forest...")

# Evaluasi standar
rf_results, rf_cm, rf_report = evaluate_model(
    y_test, y_pred_rf, "Random Forest", rf_time
)

# Print hasil standar
print_results(rf_results)

# Print detailed report per category
rf_detailed = print_detailed_report(y_test, y_pred_rf, "Random Forest", le)

# Simpan ke all_results
all_results.append(rf_results)

# Simpan model dan hasil
print("\nMenyimpan model Random Forest...")
with open(os.path.join(MODELS_DIR, 'baseline_rf.pkl'), 'wb') as f:
    pickle.dump(rf_model, f)

np.save(os.path.join(RESULTS_DIR, 'cm_rf.npy'), rf_cm)

with open(os.path.join(RESULTS_DIR, 'report_rf.json'), 'w') as f:
    json.dump(rf_report, f, indent=4)

print("Model dan hasil Random Forest tersimpan!")

# Clear memory
del y_pred_rf
clear_memory()

print_memory_status("Setelah Cleanup RF")


Evaluasi Random Forest...

HASIL EVALUASI: Random Forest (Full)
Waktu Training: 94.82 detik (1.58 menit)

Metrik Utama:
  Accuracy          : 0.9924

Precision:
  Macro             : 0.9949
  Weighted          : 0.9925
  Micro             : 0.9924

Recall:
  Macro             : 0.9932
  Weighted          : 0.9924
  Micro             : 0.9924

F1-Score:
  Macro             : 0.9940
  Weighted          : 0.9924
  Micro             : 0.9924

MCC                 : 0.9903

DETAILED CLASSIFICATION REPORT: Random Forest

               precision    recall  f1-score   support

       Benign     0.9853    0.9999    0.9926     15734
  Brute_Force     0.9983    1.0000    0.9991       571
         DDoS     0.9997    0.9867    0.9931      6097
          DoS     0.9916    0.9961    0.9938      6362
MITM/Spoofing     0.9895    0.9985    0.9940      2750
Malware/Mirai     0.9964    0.9877    0.9920      2773
        Recon     0.9985    0.9807    0.9895     10096
    Web_Based     1.0000    0.9957    

###**2.7 Training XGBoost**

In [ ]:
print_memory_status("Sebelum Training XGBoost")

start_time = time.time()

# Initialize XGBoost dengan parameter yang disesuaikan untuk dataset CICIIoT2025
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=8,                    # Lebih shallow dari CICIoT2023 (10)
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(np.unique(y_train)),  # 8 kategori
    eval_metric='mlogloss',
    tree_method='hist',             # Coba GPU dulu
    device='cuda',                  # GPU support
    random_state=42,
    n_jobs=-1
)

print("Mulai training XGBoost...")
print("(Mencoba menggunakan GPU, akan fallback ke CPU jika tidak tersedia)")

try:
    xgb_model.fit(
        X_train,
        y_train,
        verbose=True
    )
except Exception as e:
    print(f"\n GPU training failed: {e}")
    print("Switching to CPU...")

    # Fallback ke CPU
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softmax',
        num_class=len(np.unique(y_train)),
        eval_metric='mlogloss',
        tree_method='hist',
        device='cpu',               # CPU fallback
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(
        X_train,
        y_train,
        verbose=True
    )

xgb_time = time.time() - start_time

print(f"\n Training selesai dalam {xgb_time:.2f} detik ({xgb_time/60:.2f} menit)")

print("\nMelakukan prediksi pada test set...")
y_pred_xgb = xgb_model.predict(X_test)

print(" Prediksi selesai!")

print_memory_status("Setelah Training XGBoost")


[Memory Status - Sebelum Training XGBoost]
  Used: 2.54 GB / 12.67 GB (20.1%)
  Available: 10.13 GB
Mulai training XGBoost...
(Mencoba menggunakan GPU, akan fallback ke CPU jika tidak tersedia)

 Training selesai dalam 11.20 detik (0.19 menit)

Melakukan prediksi pada test set...
 Prediksi selesai!

[Memory Status - Setelah Training XGBoost]
  Used: 2.77 GB / 12.67 GB (21.9%)
  Available: 9.90 GB


In [ ]:
print("\nEvaluasi XGBoost...")

# Evaluasi standar
xgb_results, xgb_cm, xgb_report = evaluate_model(
    y_test, y_pred_xgb, "XGBoost", xgb_time
)

# Print hasil standar
print_results(xgb_results)

# Print detailed report per category
xgb_detailed = print_detailed_report(y_test, y_pred_xgb, "XGBoost", le)

# Simpan ke all_results
all_results.append(xgb_results)

# Simpan model dan hasil
print("\nMenyimpan model XGBoost...")
xgb_model.save_model(os.path.join(MODELS_DIR, 'baseline_xgboost.json'))

np.save(os.path.join(RESULTS_DIR, 'cm_xgboost.npy'), xgb_cm)

with open(os.path.join(RESULTS_DIR, 'report_xgboost.json'), 'w') as f:
    json.dump(xgb_report, f, indent=4)

print(" Model dan hasil XGBoost tersimpan!")

# Clear memory
del y_pred_xgb
clear_memory()

print_memory_status("Setelah Cleanup XGBoost")


Evaluasi XGBoost...

HASIL EVALUASI: XGBoost (Full)
Waktu Training: 11.20 detik (0.19 menit)

Metrik Utama:
  Accuracy          : 0.9951

Precision:
  Macro             : 0.9969
  Weighted          : 0.9951
  Micro             : 0.9951

Recall:
  Macro             : 0.9958
  Weighted          : 0.9951
  Micro             : 0.9951

F1-Score:
  Macro             : 0.9964
  Weighted          : 0.9951
  Micro             : 0.9951

MCC                 : 0.9938

DETAILED CLASSIFICATION REPORT: XGBoost

               precision    recall  f1-score   support

       Benign     0.9915    0.9989    0.9952     15734
  Brute_Force     0.9983    1.0000    0.9991       571
         DDoS     0.9997    0.9913    0.9955      6097
          DoS     0.9937    0.9983    0.9960      6362
MITM/Spoofing     0.9964    0.9989    0.9976      2750
Malware/Mirai     0.9989    0.9942    0.9966      2773
        Recon     0.9969    0.9884    0.9926     10096
    Web_Based     1.0000    0.9968    0.9984       936



###**2.8 Training LightGBM**

In [ ]:
print_memory_status("Sebelum Training LightGBM")

# Callback untuk logging progress
class LightGBMCallback:
    def __init__(self, total_iterations):
        self.total_iterations = total_iterations
        self.current_iteration = 0
        self.start_time = time.time()

    def __call__(self, env):
        self.current_iteration += 1
        if self.current_iteration % 10 == 0 or self.current_iteration == self.total_iterations:
            progress = (self.current_iteration / self.total_iterations) * 100
            elapsed = time.time() - self.start_time
            used, total, available, percent = get_memory_usage()
            print(f"  Iteration {self.current_iteration}/{self.total_iterations} "
                  f"({progress:.1f}%) - "
                  f"Time: {elapsed:.1f}s - "
                  f"RAM: {percent:.1f}% used ({available:.1f} GB available)")

lgb_callback = LightGBMCallback(100)

start_time = time.time()

# Initialize LightGBM dengan parameter yang disesuaikan untuk dataset CICIIoT2025
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=8,                    # Lebih shallow dari CICIoT2023 (10)
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multiclass',
    num_class=len(np.unique(y_train)),  # 8 kategori
    device='gpu',                   # Coba GPU dulu
    random_state=42,
    n_jobs=-1,
    verbose=-1                      # Suppress default logging
)

print("Mulai training LightGBM...")
print("(Mencoba menggunakan GPU, akan fallback ke CPU jika tidak tersedia)")
print("\nProgress akan ditampilkan setiap 10 iterasi")

try:
    lgb_model.fit(
        X_train,
        y_train,
        callbacks=[lgb_callback]
    )
except Exception as e:
    print(f"\n GPU training failed: {e}")
    print("Switching to CPU...")

    # Fallback ke CPU
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multiclass',
        num_class=len(np.unique(y_train)),
        device='cpu',               # CPU fallback
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    # Reset callback
    lgb_callback = LightGBMCallback(100)

    lgb_model.fit(
        X_train,
        y_train,
        callbacks=[lgb_callback]
    )

lgb_time = time.time() - start_time

print(f"\nTraining selesai dalam {lgb_time:.2f} detik ({lgb_time/60:.2f} menit)")

print("\nMelakukan prediksi pada test set...")
y_pred_lgb = lgb_model.predict(X_test)

print("Prediksi selesai!")

print_memory_status("Setelah Training LightGBM")


[Memory Status - Sebelum Training LightGBM]
  Used: 3.02 GB / 12.67 GB (23.9%)
  Available: 9.65 GB
Mulai training LightGBM...
(Mencoba menggunakan GPU, akan fallback ke CPU jika tidak tersedia)

Progress akan ditampilkan setiap 10 iterasi
  Iteration 10/100 (10.0%) - Time: 4.2s - RAM: 24.4% used (9.6 GB available)
  Iteration 20/100 (20.0%) - Time: 8.5s - RAM: 24.6% used (9.6 GB available)
  Iteration 30/100 (30.0%) - Time: 10.6s - RAM: 24.6% used (9.6 GB available)
  Iteration 40/100 (40.0%) - Time: 12.6s - RAM: 24.6% used (9.6 GB available)
  Iteration 50/100 (50.0%) - Time: 14.7s - RAM: 24.6% used (9.6 GB available)
  Iteration 60/100 (60.0%) - Time: 16.8s - RAM: 24.6% used (9.6 GB available)
  Iteration 70/100 (70.0%) - Time: 20.7s - RAM: 24.5% used (9.6 GB available)
  Iteration 80/100 (80.0%) - Time: 23.2s - RAM: 24.6% used (9.6 GB available)
  Iteration 90/100 (90.0%) - Time: 25.3s - RAM: 24.6% used (9.6 GB available)

 GPU training failed: Check failed: (best_split_info.left_

In [ ]:
print("\nEvaluasi LightGBM...")

# Evaluasi standar
lgb_results, lgb_cm, lgb_report = evaluate_model(
    y_test, y_pred_lgb, "LightGBM", lgb_time
)

# Print hasil standar
print_results(lgb_results)

# Print detailed report per category
lgb_detailed = print_detailed_report(y_test, y_pred_lgb, "LightGBM", le)

# Simpan ke all_results
all_results.append(lgb_results)

# Simpan model dan hasil
print("\nMenyimpan model LightGBM...")
with open(os.path.join(MODELS_DIR, 'baseline_lightgbm.pkl'), 'wb') as f:
    pickle.dump(lgb_model, f)

np.save(os.path.join(RESULTS_DIR, 'cm_lightgbm.npy'), lgb_cm)

with open(os.path.join(RESULTS_DIR, 'report_lightgbm.json'), 'w') as f:
    json.dump(lgb_report, f, indent=4)

print(" Model dan hasil LightGBM tersimpan!")

# Clear memory
del y_pred_lgb
clear_memory()

print_memory_status("Setelah Cleanup LightGBM")


Evaluasi LightGBM...

HASIL EVALUASI: LightGBM (Full)
Waktu Training: 107.15 detik (1.79 menit)

Metrik Utama:
  Accuracy          : 0.9950

Precision:
  Macro             : 0.9968
  Weighted          : 0.9950
  Micro             : 0.9950

Recall:
  Macro             : 0.9905
  Weighted          : 0.9950
  Micro             : 0.9950

F1-Score:
  Macro             : 0.9936
  Weighted          : 0.9950
  Micro             : 0.9950

MCC                 : 0.9936

DETAILED CLASSIFICATION REPORT: LightGBM

               precision    recall  f1-score   support

       Benign     0.9921    0.9989    0.9955     15734
  Brute_Force     0.9982    0.9527    0.9749       571
         DDoS     0.9984    0.9926    0.9955      6097
          DoS     0.9939    0.9984    0.9962      6362
MITM/Spoofing     0.9971    0.9993    0.9982      2750
Malware/Mirai     0.9993    0.9953    0.9973      2773
        Recon     0.9958    0.9890    0.9924     10096
    Web_Based     1.0000    0.9979    0.9989       9

###**2.9 Training Deep Neural Network (DNN)**

In [ ]:
print_memory_status("Sebelum Training DNN")

# Clear memory sebelum training
clear_memory()

# Get number of features and classes
num_features = X_train.shape[1]
num_classes = len(np.unique(y_train))

print(f"\nParameter DNN:")
print(f"  - Input features: {num_features}")
print(f"  - Output classes: {num_classes}")
print(f"  - Train samples: {X_train.shape[0]:,}")

def create_dnn_model(num_features, num_classes):
    """
    Membuat arsitektur DNN untuk 8 Kategori CICIIoT2025
    """
    model = keras.Sequential([
        layers.Input(shape=(num_features,)),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.BatchNormalization(),

        layers.Dense(16, activation='relu'),

        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Create model
dnn_model = create_dnn_model(num_features, num_classes)

print("\nArsitektur DNN:")
dnn_model.summary()

# Custom callback untuk logging memory
class MemoryLoggingCallback(keras.callbacks.Callback):
    def __init__(self, log_frequency=1):
        super().__init__()
        self.log_frequency = log_frequency
        self.start_time = None

    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        print("\n✓ Training dimulai...")
        print_memory_status("Train Begin")

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.log_frequency == 0:
            elapsed = time.time() - self.start_time
            used, total, available, percent = get_memory_usage()
            print(f"\nEpoch {epoch + 1}:")
            print(f"  Loss: {logs.get('loss'):.4f} - Acc: {logs.get('accuracy'):.4f}")
            print(f"  Val Loss: {logs.get('val_loss'):.4f} - Val Acc: {logs.get('val_accuracy'):.4f}")
            print(f"  Time: {elapsed:.1f}s - RAM: {percent:.1f}% ({available:.1f} GB available)")

# Callbacks
checkpoint = ModelCheckpoint(
    os.path.join(CHECKPOINT_DIR, 'dnn_best.keras'),
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

memory_logger = MemoryLoggingCallback(log_frequency=1)

print("\nMulai training DNN...")
print("Parameters:")
print("  - Epochs: 50")
print("  - Batch size: 256")
print("  - Validation split: 10% (stratified from train)")
print("  - Early stopping patience: 5")

start_time = time.time()

try:
    history = dnn_model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=256,
        callbacks=[checkpoint, early_stopping, reduce_lr, memory_logger],
        verbose=0  # Suppress default output, pakai custom callback
    )

    dnn_time = time.time() - start_time

    print(f"\n Training selesai dalam {dnn_time:.2f} detik ({dnn_time/60:.2f} menit)")
    print_memory_status("Setelah Training DNN")

except Exception as e:
    print(f"\n Error saat training DNN: {e}")
    print("Jika error memory, coba kurangi batch_size atau epochs")
    raise


[Memory Status - Sebelum Training DNN]
  Used: 3.02 GB / 12.67 GB (23.8%)
  Available: 9.65 GB
  Memory cleared

Parameter DNN:
  - Input features: 71
  - Output classes: 8
  - Train samples: 386,274

Arsitektur DNN:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           136 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,112 (82.47 KB)

 Trainable params: 20,664 (80.72 KB)

 Non-trainable params: 448 (1.75 KB)


Mulai training DNN...
Parameters:
  - Epochs: 50
  - Batch size: 256
  - Validation split: 10% (stratified from train)
  - Early stopping patience: 5

✓ Training dimulai...

[Memory Status - Train Begin]
  Used: 3.30 GB / 12.67 GB (26.1%)
  Available: 9.37 GB

Epoch 1: val_loss improved from inf to 0.27138, saving model to /content/drive/My Drive/CICIIoT2025/Baseline8Multiclass/Checkpoints/dnn_best.keras

Epoch 1:
  Loss: 0.5595 - Acc: 0.8275
  Val Loss: 0.2714 - Val Acc: 0.9132
  Time: 15.5s - RAM: 24.9% (9.5 GB available)

Epoch 2: val_loss improved from 0.27138 to 0.17765, saving model to /content/drive/My Drive/CICIIoT2025/Baseline8Multiclass/Checkpoints/dnn_best.keras

Epoch 2:
  Loss: 0.3276 - Acc: 0.9020
  Val Loss: 0.1776 - Val Acc: 0.9450
  Time: 20.2s - RAM: 24.9% (9.5 GB available)

Epoch 3: val_loss improved from 0.17765 to 0.15598, saving model to /content/drive/My Drive/CICIIoT2025/Baseline8Multiclass/Checkpoints/dnn_best.keras

Epoch 3:
  Loss: 0.2608 - Acc: 0.9208
  Va

In [ ]:
print("\nMelakukan prediksi pada test set...")

# Prediksi
y_pred_dnn_proba = dnn_model.predict(X_test, batch_size=256, verbose=0)
y_pred_dnn = np.argmax(y_pred_dnn_proba, axis=1)

print(" Prediksi selesai!")

print("\nEvaluasi Deep Neural Network...")

# Evaluasi standar
dnn_results, dnn_cm, dnn_report = evaluate_model(
    y_test, y_pred_dnn, "Deep Neural Network", dnn_time
)

# Print hasil standar
print_results(dnn_results)

# Print detailed report per category
dnn_detailed = print_detailed_report(y_test, y_pred_dnn, "Deep Neural Network", le)

# Simpan ke all_results
all_results.append(dnn_results)

# Simpan model dan hasil
print("\nMenyimpan model DNN...")
dnn_model.save(os.path.join(MODELS_DIR, 'baseline_dnn.keras'))

np.save(os.path.join(RESULTS_DIR, 'cm_dnn.npy'), dnn_cm)

with open(os.path.join(RESULTS_DIR, 'report_dnn.json'), 'w') as f:
    json.dump(dnn_report, f, indent=4)

# Save training history
history_dict = {
    'loss': [float(x) for x in history.history['loss']],
    'accuracy': [float(x) for x in history.history['accuracy']],
    'val_loss': [float(x) for x in history.history['val_loss']],
    'val_accuracy': [float(x) for x in history.history['val_accuracy']]
}

with open(os.path.join(RESULTS_DIR, 'dnn_history.json'), 'w') as f:
    json.dump(history_dict, f, indent=4)

print(" Model dan hasil DNN tersimpan!")

# Clear memory
del y_pred_dnn_proba, y_pred_dnn
clear_memory()

print_memory_status("Setelah Cleanup DNN")


Melakukan prediksi pada test set...
 Prediksi selesai!

Evaluasi Deep Neural Network...

HASIL EVALUASI: Deep Neural Network (Full)
Waktu Training: 236.72 detik (3.95 menit)

Metrik Utama:
  Accuracy          : 0.9747

Precision:
  Macro             : 0.9695
  Weighted          : 0.9748
  Micro             : 0.9747

Recall:
  Macro             : 0.9595
  Weighted          : 0.9747
  Micro             : 0.9747

F1-Score:
  Macro             : 0.9642
  Weighted          : 0.9747
  Micro             : 0.9747

MCC                 : 0.9678

DETAILED CLASSIFICATION REPORT: Deep Neural Network

               precision    recall  f1-score   support

       Benign     0.9840    0.9962    0.9901     15734
  Brute_Force     0.9468    0.8722    0.9079       571
         DDoS     0.9557    0.9634    0.9596      6097
          DoS     0.9750    0.9398    0.9571      6362
MITM/Spoofing     0.9241    0.9611    0.9422      2750
Malware/Mirai     0.9904    0.9704    0.9803      2773
        Recon     

###**2.10 Perbandingan Hasil Semua Model**

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame(all_results)

# Sort by accuracy descending
comparison_df = comparison_df.sort_values('accuracy', ascending=False)

print("\n" + "="*70)
print("COMPREHENSIVE METRICS COMPARISON")
print("="*70)

print("\n", comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv(
    os.path.join(RESULTS_DIR, 'baseline_comparison.csv'),
    index=False
)

print(f"\n Comparison table saved to: {RESULTS_DIR}")

# Verify no NaN values
print("\n" + "="*70)
print("VERIFICATION: Checking for NaN values...")
print("="*70)

nan_counts = comparison_df.isnull().sum()
if nan_counts.sum() == 0:
    print("   SUCCESS: No NaN values found!")
else:
    print("   WARNING: NaN values detected:")
    print(nan_counts[nan_counts > 0])

# Print best models per metric
print("\n" + "="*70)
print("MODEL TERBAIK PER METRIK")
print("="*70)

metrics_to_check = {
    'Accuracy': 'accuracy',
    'F1-Score (Macro)': 'f1_macro',
    'F1-Score (Weighted)': 'f1_weighted',
    'Precision (Macro)': 'precision_macro',
    'Recall (Macro)': 'recall_macro',
    'MCC': 'mcc'
}

for metric_name, metric_col in metrics_to_check.items():
    best_idx = comparison_df[metric_col].idxmax()
    best_row = comparison_df.loc[best_idx]
    print(f"\n{metric_name}:")
    print(f"  Model            : {best_row['model_name']}")
    print(f"  Score            : {best_row[metric_col]:.4f}")
    print(f"  Training Time    : {best_row['training_time_seconds']:.2f}s ({best_row['training_time_seconds']/60:.2f} min)")


COMPREHENSIVE METRICS COMPARISON

          model_name dataset_type  training_time_seconds  accuracy  precision_macro  precision_weighted  precision_micro  recall_macro  recall_weighted  recall_micro  f1_macro  f1_weighted  f1_micro      mcc
            XGBoost         Full              11.199870  0.995123         0.996925            0.995143         0.995123      0.995847         0.995123      0.995123  0.996378     0.995121  0.995123 0.993785
           LightGBM         Full             107.146287  0.994991         0.996838            0.995005         0.994991      0.990517         0.994991      0.994991  0.993606     0.994983  0.994991 0.993614
           LightGBM         Full              37.174346  0.994815         0.996510            0.994823         0.994815      0.990329         0.994815      0.994815  0.993345     0.994805  0.994815 0.993387
      Random Forest         Full              94.820435  0.992387         0.994902            0.992465         0.992387      0.993169   

In [ ]:
print("\n" + "="*70)
print("DETAILED METRICS TABLE")
print("="*70)

# Create detailed table with important metrics
detailed_metrics = comparison_df[[
    'model_name',
    'training_time_seconds',
    'accuracy',
    'precision_macro',
    'precision_weighted',
    'recall_macro',
    'recall_weighted',
    'f1_macro',
    'f1_weighted',
    'mcc'
]].copy()

# Rename columns for better readability
detailed_metrics.columns = [
    'Model',
    'Time (s)',
    'Accuracy',
    'Precision (Macro)',
    'Precision (Weighted)',
    'Recall (Macro)',
    'Recall (Weighted)',
    'F1 (Macro)',
    'F1 (Weighted)',
    'MCC'
]

# Format time
detailed_metrics['Time (s)'] = detailed_metrics['Time (s)'].apply(lambda x: f"{x:.2f}")

# Format metrics to 4 decimal places
for col in detailed_metrics.columns[2:]:
    detailed_metrics[col] = detailed_metrics[col].apply(lambda x: f"{x:.4f}")

print("\n", detailed_metrics.to_string(index=False))

# Save detailed metrics
detailed_metrics.to_csv(
    os.path.join(RESULTS_DIR, 'detailed_metrics.csv'),
    index=False
)

print(f"\n Detailed metrics saved!")


DETAILED METRICS TABLE

               Model Time (s) Accuracy Precision (Macro) Precision (Weighted) Recall (Macro) Recall (Weighted) F1 (Macro) F1 (Weighted)    MCC
            XGBoost    11.20   0.9951            0.9969               0.9951         0.9958            0.9951     0.9964        0.9951 0.9938
           LightGBM   107.15   0.9950            0.9968               0.9950         0.9905            0.9950     0.9936        0.9950 0.9936
           LightGBM    37.17   0.9948            0.9965               0.9948         0.9903            0.9948     0.9933        0.9948 0.9934
      Random Forest    94.82   0.9924            0.9949               0.9925         0.9932            0.9924     0.9940        0.9924 0.9903
Deep Neural Network   236.72   0.9747            0.9695               0.9748         0.9595            0.9747     0.9642        0.9747 0.9678

 Detailed metrics saved!
